# India Structure Data Pipeline

Build city-level structure polygons for Indian cities by combining:

- **Overture Maps buildings** for primary polygons and building attributes.
- **Microsoft Global ML Building Footprints** as a polygon gap-filler and height/confidence source.
- **OSMnx / OpenStreetMap** for building tags, names, floors, units, and heights where mapped.
- **Optional NSI and census fields**, disabled by default for India.
- **Optional parcel files or public ArcGIS parcel layers** for parcel ID, address, land use, zoning, owner, value, and year-built context.

The output is one row per structure polygon with raw source fields collapsed into auditable columns such as `StructureTypeSource`, `NumStoriesSource`, `HeightSource`, `ParcelSource`, and `ParcelMatchMethod`.


## Install Dependencies

Run this once if the environment is missing packages.

```python
%pip install -r requirements.txt
```


In [ ]:
import importlib
import sys
from pathlib import Path

cwd = Path.cwd()
if (cwd / "structure_pipeline.py").exists() and cwd.name == "India":
    PIPELINE_DIR = cwd
elif (cwd / "India" / "structure_pipeline.py").exists():
    PIPELINE_DIR = cwd / "India"
else:
    PIPELINE_DIR = Path("India").resolve()

sys.path.insert(0, str(PIPELINE_DIR))
sys.modules.pop("structure_pipeline", None)
import structure_pipeline

structure_pipeline = importlib.reload(structure_pipeline)
from structure_pipeline import PipelineConfig, build_many_cities

## Configure Cities and Sources

Add as many cities as needed. For large cities, the first run can take time because OSM, Overture, Microsoft, and optional parcel calls are network-bound or file-size-bound. Local source files are reused on later runs.

Parcel data is local-source specific. Add parcel files or public parcel services only when you have a reliable source for that city.


In [ ]:
CITIES = [
    {"city": "Chennai", "state": "Tamil Nadu"},
    # {"city": "Mumbai", "state": "Maharashtra"},
    # {"city": "Bengaluru", "state": "Karnataka"},
    # Example with a local parcel file:
    # {
    #     "city": "Chennai",
    #     "state": "Tamil Nadu",
    #     "parcel_source": {
    #         "path": "data/raw/chennai_parcels.gpkg",
    #         "field_map": {
    #             "ParcelID": "property_id",
    #             "ParcelAddress": "address",
    #             "ParcelLandUse": "land_use",
    #             "ParcelZoning": "zoning",
    #             "ParcelOwner": "owner_name",
    #             "ParcelAssessedValue": "assessed_value",
    #             "ParcelYearBuilt": "year_built",
    #         },
    #     },
    # },
]

config = PipelineConfig(
    data_dir=PIPELINE_DIR / "data",
    output_dir=PIPELINE_DIR / "data/output",
    raw_dir=PIPELINE_DIR / "data/raw",
    cache_dir=PIPELINE_DIR / "cache",
    country="India",
    download_missing=True,
    use_overture=True,
    use_microsoft=True,
    use_osm=True,
    use_nsi=False,
    use_census=False,
    use_parcels=True,
    # Keep these False while exploring; turn them on for production runs.
    strict_sources=False,
    fail_on_empty_output=False,
    write_run_metadata=True,
    add_microsoft_unmatched=True,
    parcel_sources={
        # "chennai_tamil_nadu_india": {
        #     "url": "https://.../FeatureServer/0",
        #     "field_map": {"ParcelID": "PARCEL_ID"},
        # },
    },
    nsi_tile_size_deg=0.08,
)


## Run Pipeline

Outputs are written to `data/output/{city}_{state}_india_structures.parquet` and a combined `data/output/structures_master.parquet`.

In [ ]:
structures = build_many_cities(CITIES, config)
structures.head()

## Inspect Coverage

These checks show how often the final fields came from each source or inference method.

In [ ]:
print("Rows:", len(structures))
print("Cities:", structures[["City", "State"]].drop_duplicates().to_dict("records"))

summary_cols = [
    "FootprintSource",
    "StructureTypeSource",
    "NumUnitsSource",
    "NumStoriesSource",
    "HeightSource",
    "OccupantCountMethod",
    "ParcelSource",
    "ParcelMatchMethod",
]
for col in summary_cols:
    print(f"\n{col}")
    print(structures[col].value_counts(dropna=False).head(20))


## Output Schema

Key normalized columns:

- `geometry`: structure polygon in EPSG:4326.
- `StructureType`: normalized type such as `residential`, `condo`, `apartment`, `commercial`, `hotel`, `garage`, `barn`, `industrial`, `warehouse`, `education`, `healthcare`, or `unknown`.
- `StructureTypeRaw` and `StructureTypeSource`: raw value and source used to derive `StructureType`.
- `BuildingName` and `BuildingNameSource`: available building name and the source used for it.
- `NumUnits` and `NumUnitsSource`: explicit OSM residential units, NSI units when enabled, or `inferred_single_family` when the source type clearly describes a single-family structure.
- `NumStories` and `NumStoriesSource`: OSM floors, Overture floors, NSI stories when enabled, or a height-derived estimate.
- `HeightM` and `HeightSource`: source-derived building height in meters.
- `OccupantCount` and `OccupantCountMethod`: usually empty for India unless you enable or add an authoritative source later.
- `ParcelID`, `ParcelAddress`, `ParcelLandUse`, `ParcelZoning`, `ParcelOwner`, `ParcelAssessedValue`, `ParcelYearBuilt`, and `ParcelArea_m2`: parcel context when a parcel source is configured.
- `ParcelMatchMethod`: currently `representative_point_within`, meaning the structure's representative point fell inside the parcel polygon.

Occupant counts should be treated as estimates unless your downstream workflow has a stronger authoritative source.


In [ ]:
columns = [
    "StructureID", "City", "State", "FootprintSource", "StructureType",
    "StructureTypeRaw", "StructureTypeSource", "BuildingName", "BuildingNameSource",
    "NumUnits", "NumUnitsSource", "NumStories", "NumStoriesSource",
    "HeightM", "HeightSource", "OccupantCount", "OccupantCountMethod",
    "ParcelID", "ParcelAddress", "ParcelLandUse", "ParcelZoning",
    "ParcelAssessedValue", "ParcelYearBuilt", "ParcelArea_m2",
    "ParcelMatchMethod", "FootprintArea_m2", "geometry",
]
structures[columns].head(100)
